In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [6]:
!pip install dagshub mlflow xgboost -q
!pip install mlflow
!pip install dagshub

import xgboost as xgb
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import dagshub
import warnings
import json
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from mlflow.tracking import MlflowClient

dagshub.init(repo_owner='akave23', repo_name='IEEE-CIS-Fraud-Detection-ML2', mlflow=True)

print(f'MLflow URI: {mlflow.get_tracking_uri()}')


Initialized MLflow to track repo "akave23/IEEE-CIS-Fraud-Detection-ML2"

Repository akave23/IEEE-CIS-Fraud-Detection-ML2 initialized!

MLflow URI: https://dagshub.com/akave23/IEEE-CIS-Fraud-Detection-ML2.mlflow


In [7]:
client = MlflowClient()

experiments_to_check = [
    ('XGBoost_Training_v1',      'xgb_pipeline'),
    ('RandomForest_Training_v1', 'rf_pipeline'),
]

all_final_runs = []

for exp_name, artifact_path in experiments_to_check:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f'  SKIP: experiment "{exp_name}" not found')
        continue

    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string="tags.mlflow.runName LIKE '%Final%'",
        order_by=['metrics.auc_val_holdout DESC'],
        max_results=1
    )
    if not runs:
        print(f'  SKIP: no Final run in "{exp_name}"')
        continue

    r       = runs[0]
    val_auc = r.data.metrics.get('auc_val_holdout', 0)
    pr_auc  = r.data.metrics.get('prauc_val', r.data.metrics.get('auc_pipeline_val', 0))

    all_final_runs.append({
        'experiment':    exp_name,
        'run_id':        r.info.run_id,
        'artifact_path': artifact_path,
        'val_auc':       val_auc,
        'pr_auc':        pr_auc,
    })
    print(f'{exp_name:35s}  Val AUC={val_auc:.4f}  PR-AUC={pr_auc:.4f}')

compare_df = pd.DataFrame(all_final_runs).sort_values('val_auc', ascending=False)
print('\n=== Final Comparison ===')
print(compare_df[['experiment','val_auc','pr_auc']].to_string(index=False))

BEST = compare_df.iloc[0]
print(f'\nBest: {BEST["experiment"]}  run_id={BEST["run_id"][:8]}  Val AUC={BEST["val_auc"]:.4f}')

XGBoost_Training_v1                  Val AUC=0.9709  PR-AUC=0.9647
RandomForest_Training_v1             Val AUC=0.9127  PR-AUC=0.9072

=== Final Comparison ===
              experiment  val_auc   pr_auc
     XGBoost_Training_v1 0.970858 0.964677
RandomForest_Training_v1 0.912656 0.907170

Best: XGBoost_Training_v1  run_id=b4b57e7a  Val AUC=0.9709


In [16]:
from mlflow.tracking import MlflowClient
import mlflow

MODEL_NAME = "fraud-detection-xgb"

model_uri = "mlflow-artifacts:/7553e5d688374baea357ce5b93892efb/models/m-db004c3e1dce45bd983938719712d7ad/artifacts"

print(model_uri)

registered = mlflow.register_model(
    model_uri=model_uri,
    name=MODEL_NAME
)

client = MlflowClient()

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Production",
    archive_existing_versions=True
)

Registered model 'fraud-detection-xgb' already exists. Creating a new version of this model...
2026/05/06 08:46:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: fraud-detection-xgb, version 1


mlflow-artifacts:/7553e5d688374baea357ce5b93892efb/models/m-db004c3e1dce45bd983938719712d7ad/artifacts


Created version '1' of model 'fraud-detection-xgb'.


<ModelVersion: aliases=[], creation_timestamp=1778057168743, current_stage='Production', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1778057168929, metrics=None, model_id=None, name='fraud-detection-xgb', params=None, run_id='', run_link='', source='mlflow-artifacts:/7553e5d688374baea357ce5b93892efb/models/m-db004c3e1dce45bd983938719712d7ad/artifacts', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [18]:
model_uri = "mlflow-artifacts:/7553e5d688374baea357ce5b93892efb/models/m-db004c3e1dce45bd983938719712d7ad/artifacts"

loaded_pipeline = mlflow.sklearn.load_model(model_uri)

In [19]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv').merge(
    pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv'),
    on='TransactionID', how='left')

y = train['isFraud'].values
X = train.drop(['isFraud', 'TransactionID'], axis=1)


_, X_valid, _, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

val_preds = loaded_pipeline.predict_proba(X_valid)[:, 1]
val_auc   = roc_auc_score(y_valid, val_preds)
val_prauc = average_precision_score(y_valid, val_preds)

print(f'Validation on raw X_valid:')
print(f'  ROC-AUC : {val_auc:.4f}')
print(f'  PR-AUC  : {val_prauc:.4f}')

Validation on raw X_valid:
  ROC-AUC : 0.9647
  PR-AUC  : 0.8178


In [20]:
test = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv').merge(
    pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv'),
    on='TransactionID', how='left')

test_ids = test['TransactionID'].values
X_test   = test.drop(columns=['TransactionID'])

print(f'Test: {X_test.shape}')

# Pipeline handles all preprocessing internally
test_preds = loaded_pipeline.predict_proba(X_test)[:, 1]

print(f'Predicted fraud rate: {test_preds.mean():.4f}')
print(f'Min={test_preds.min():.4f} | Max={test_preds.max():.4f}')

submission = pd.DataFrame({'TransactionID': test_ids, 'isFraud': test_preds})
submission.to_csv('submission.csv', index=False)

assert submission['isFraud'].between(0, 1).all()
assert not submission['isFraud'].isna().any()
assert len(submission) == len(test)
print(f'\nsubmission.csv saved — {len(submission):,} rows.')
print(submission.head())

Test: (506691, 432)
Predicted fraud rate: 0.0241
Min=0.0000 | Max=0.9997

submission.csv saved — 506,691 rows.
   TransactionID   isFraud
0        3663549  0.004117
1        3663550  0.000274
2        3663551  0.000841
3        3663552  0.001120
4        3663553  0.000127


In [21]:
mlflow.set_experiment('Model_Inference')

with mlflow.start_run(run_name='BestModel_Inference') as run:
    mlflow.log_param('model_name',     MODEL_NAME)
    mlflow.log_param('model_source',   BEST['experiment'])
    mlflow.log_param('registry_uri',   registry_uri)
    mlflow.log_param('source_run_id',  BEST['run_id'])
    mlflow.log_metric('val_roc_auc',   val_auc)
    mlflow.log_metric('val_pr_auc',    val_prauc)
    mlflow.log_metric('test_samples',  len(test_preds))
    mlflow.log_metric('fraud_rate',    float(test_preds.mean()))
    mlflow.log_artifact('submission.csv')
    print(f'Logged. Run ID: {run.info.run_id}')

print(f'\nModel : {MODEL_NAME} (Production)')
print(f'Source: {BEST["experiment"]}')
print(f'Val ROC-AUC: {val_auc:.4f}')
print(f'Val PR-AUC : {val_prauc:.4f}')

2026/05/06 08:56:40 INFO mlflow.tracking.fluent: Experiment with name 'Model_Inference' does not exist. Creating a new experiment.


Logged. Run ID: 0175d689c2b4421e916a6090607c7849
🏃 View run BestModel_Inference at: https://dagshub.com/akave23/IEEE-CIS-Fraud-Detection-ML2.mlflow/#/experiments/4/runs/0175d689c2b4421e916a6090607c7849
🧪 View experiment at: https://dagshub.com/akave23/IEEE-CIS-Fraud-Detection-ML2.mlflow/#/experiments/4

Model : fraud-detection-xgb (Production)
Source: XGBoost_Training_v1
Val ROC-AUC: 0.9647
Val PR-AUC : 0.8178
